In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PolicyNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        logits = self.fc3(x)
        return torch.softmax(logits, dim=-1)

env = gym.make('CartPole-v1')

input_dim = env.observation_space.shape[0]  # 4
output_dim = env.action_space.n             # 2
hidden_dim = 128
lr = 0.001
gamma = 0.99
num_episodes = 1000

policy_net = PolicyNetwork(input_dim, hidden_dim, output_dim).to(device)
optimizer = optim.Adam(policy_net.parameters(), lr=lr)

def compute_returns(rewards, gamma):
    R = 0
    returns = []
    for r in reversed(rewards):
        R = r + gamma * R
        returns.insert(0, R)
    return returns

for episode in range(num_episodes):
    state = env.reset()[0]
    log_probs = []
    rewards = []

    done = False
    total_reward = 0

    while not done:
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        probs = policy_net(state_tensor)
        action_distribution = torch.distributions.Categorical(probs)
        action = action_distribution.sample()
        
        log_prob = action_distribution.log_prob(action)
        log_probs.append(log_prob)

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated
        state = next_state

        rewards.append(reward)
        total_reward += reward

    returns = compute_returns(rewards, gamma)
    returns = torch.tensor(returns).to(device)

    returns = (returns - returns.mean()) / (returns.std() + 1e-9)

    policy_loss = []
    for log_prob, G in zip(log_probs, returns):
        policy_loss.append(-log_prob * G)

    optimizer.zero_grad()
    loss = torch.stack(policy_loss).sum()
    loss.backward()
    optimizer.step()

    if episode % 50 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward}, Loss: {loss.item():.4f}")

print("Training finished.")

ModuleNotFoundError: No module named 'gym'